# Black-Scholes validation

This notebook compares Monte Carlo European call and put prices against analytical Black-Scholes values across strikes, volatilities, and maturities.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from mc_options.black_scholes import black_scholes_price
from mc_options.models import MarketParameters, OptionParameters, SimulationParameters
from mc_options.pricer import price_option

In [2]:
rows = []
for strike in [80, 90, 100, 110, 120]:
    for sigma in [0.10, 0.20, 0.40]:
        for maturity in [0.25, 0.5, 1.0, 2.0]:
            market = MarketParameters(spot=100, rate=0.03, volatility=sigma, maturity=maturity)
            for option_type in ['call', 'put']:
                option = OptionParameters(strike=strike, option_type=option_type)
                sim = SimulationParameters(num_paths=100_000, num_steps=1, seed=42)
                result = price_option(market, option, sim)
                benchmark = black_scholes_price(market, option)
                abs_error = abs(result.price - benchmark)
                rows.append({
                    'option_type': option_type,
                    'strike': strike,
                    'sigma': sigma,
                    'maturity': maturity,
                    'mc_price': result.price,
                    'bs_price': benchmark,
                    'abs_error': abs_error,
                    'rel_error_pct': 100 * abs_error / benchmark if benchmark else 0.0,
                    'ci_low': result.ci_low,
                    'ci_high': result.ci_high,
                    'ci_contains_bs': result.ci_low <= benchmark <= result.ci_high,
                })

validation = pd.DataFrame(rows)
validation.head()

,option_type,strike,sigma,maturity,mc_price,bs_price,abs_error,rel_error_pct,ci_low,ci_high,ci_contains_bs
0,call,80,0.1,0.25,20.577535,20.597757,0.020222,0.098175,20.546411,20.608660,True
1,put,80,0.1,0.25,0.000000,0.000002,0.000002,100.000000,0.000000,0.000000,False
2,call,80,0.1,0.50,21.163601,21.191661,0.028060,0.132410,21.119567,21.207634,True
3,put,80,0.1,0.50,0.000585,0.000616,0.000031,4.961086,0.000357,0.000813,True
4,call,80,0.1,1.00,22.343122,22.380354,0.037232,0.166361,22.281014,22.405229,True


In [3]:
validation.groupby('option_type').agg(
    cases=('mc_price', 'count'),
    mean_abs_error=('abs_error', 'mean'),
    max_abs_error=('abs_error', 'max'),
    ci_coverage=('ci_contains_bs', 'mean'),
    median_rel_error_pct=('rel_error_pct', 'median'),
)

,cases,mean_abs_error,max_abs_error,ci_coverage,median_rel_error_pct
option_type,,,,,
call,60,0.025492,0.077633,1.000000,0.242481
put,60,0.049238,0.166968,0.983333,0.657575
